# Model comparison and champion selection

- **Question:** Which candidate earned the champion label on future-season validation, and what did the untouched test reveal afterward?
- **Data used:** Canonical Phase 3 baseline predictions, Phase 4 learned predictions, champion records, run lineage, and the Phase 4 JSON report when present.
- **Unit of observation:** One matched player-season prediction for one position, target, and candidate.
- **Target:** Next-season fantasy points per active game, games active, or total fantasy points.
- **Feature cutoff:** Every learned row records a maximum training season strictly earlier than its prediction season.
- **Validation strategy:** Select with pooled 2020-2024 validation MAE plus a paired-bootstrap uncertainty gate; attach the frozen 2025 test after selection. Learned ties and improvements whose 95% interval crosses zero retain the best transparent baseline.
- **Interpretation caveat:** A champion is best under this target, sample, ruleset, and validation policy. It does not answer ADP timing, replacement value, roster fit, or rookie performance outside the validated population.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import duckdb
import matplotlib.pyplot as plt
import pandas as pd

from fantasy_draft_ai.models.player_projection.evaluation import regression_metrics


def find_project_root(start: Path | None = None) -> Path:
    candidate = (start or Path.cwd()).resolve()
    for directory in (candidate, *candidate.parents):
        if (directory / "pyproject.toml").is_file():
            return directory
    raise FileNotFoundError("Could not find the repository root.")


def table_exists(connection: duckdb.DuckDBPyConnection, table_name: str) -> bool:
    row = connection.execute(
        "SELECT count(*) FROM information_schema.tables WHERE table_name = ?",
        [table_name],
    ).fetchone()
    return bool(row and row[0])


def compact_report_value(value: Any) -> str | int | float | bool | None:
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    if isinstance(value, list):
        return f"list[{len(value)}]"
    if isinstance(value, dict):
        return f"dict[{len(value)}]"
    return type(value).__name__


PROJECT_ROOT = find_project_root()
WAREHOUSE_PATH = PROJECT_ROOT / "data" / "warehouse" / "fantasy_football.duckdb"
REPORT_PATH = PROJECT_ROOT / "docs" / "PHASE_4_MODEL_EVALUATION.json"
phase4_report = json.loads(REPORT_PATH.read_text(encoding="utf-8")) if REPORT_PATH.is_file() else {}
print({"warehouse_ready": WAREHOUSE_PATH.is_file(), "report_ready": bool(phase4_report)})

## Read candidates from the same warehouse

A fair comparison requires the same actual player-season rows. Baselines and learned models are labeled separately so the report never disguises a heuristic as trained or a trained candidate as automatically selected.

In [ ]:
baseline_predictions = pd.DataFrame()
learned_predictions = pd.DataFrame()
champions = pd.DataFrame()
runs = pd.DataFrame()
if WAREHOUSE_PATH.is_file():
    with duckdb.connect(str(WAREHOUSE_PATH), read_only=True) as connection:
        if table_exists(connection, "baseline_predictions"):
            baseline_predictions = connection.execute(
                """
                SELECT player_id, prediction_season, position, target_name,
                       'baseline' AS candidate_source, baseline_name AS candidate_name,
                       predicted_value, actual_value, NULL::INTEGER AS training_max_season
                FROM baseline_predictions
                WHERE actual_value IS NOT NULL AND prediction_season BETWEEN 2020 AND 2025
                """
            ).df()
        if table_exists(connection, "player_projection_predictions"):
            learned_predictions = connection.execute(
                """
                SELECT player_id, prediction_season, position, target_name,
                       'learned' AS candidate_source, model_family AS candidate_name,
                       predicted_value, actual_value, training_max_season
                FROM player_projection_predictions
                WHERE actual_value IS NOT NULL AND prediction_season BETWEEN 2020 AND 2025
                """
            ).df()
        if table_exists(connection, "player_projection_champions"):
            champions = connection.execute(
                """
                SELECT position, target_name, selected_source, selected_name,
                       selection_metric, selection_value, reference_baseline_name,
                       reference_baseline_value, improvement
                FROM player_projection_champions
                ORDER BY position, target_name
                """
            ).df()
        if table_exists(connection, "player_projection_runs"):
            runs = connection.execute(
                """
                SELECT run_id, feature_data_fingerprint, target_data_fingerprint,
                       build_fingerprint, scoring_ruleset_fingerprint,
                       baseline_report_fingerprint, model_feature_fingerprint,
                       model_config_fingerprint, status
                FROM player_projection_runs
                ORDER BY trained_at DESC
                """
            ).df()

candidates = pd.concat([baseline_predictions, learned_predictions], ignore_index=True)
print({"baseline_rows": len(baseline_predictions), "learned_rows": len(learned_predictions)})

## Audit coverage before comparing errors

Counts alone do not prove equality, but they make an immediate mismatch visible. Production champion selection additionally checks the exact player-season keys and actual values for every required candidate.

In [ ]:
if candidates.empty:
    coverage = pd.DataFrame()
    print("No complete Phase 4 comparison is stored yet.")
else:
    coverage = (
        candidates.groupby(
            ["prediction_season", "position", "target_name", "candidate_source", "candidate_name"]
        )
        .agg(rows=("player_id", "size"), unique_players=("player_id", "nunique"))
        .reset_index()
    )
    if not learned_predictions.empty:
        assert (
            learned_predictions["training_max_season"] < learned_predictions["prediction_season"]
        ).all()
    display(coverage)

## Calculate real validation and test metrics

Validation seasons are pooled because each eligible player-season is one observation in the selection metric. The test remains a separate period even when its result is surprising.

In [ ]:
comparison_rows: list[dict[str, object]] = []
if not candidates.empty:
    working = candidates.assign(
        period=lambda frame: frame["prediction_season"].map(
            lambda season: "test_2025" if season == 2025 else "validation_2020_2024"
        )
    )
    group_columns = ["position", "target_name", "candidate_source", "candidate_name", "period"]
    for keys, group in working.groupby(group_columns):
        metrics = regression_metrics(
            group["actual_value"],
            group["predicted_value"],
            top_n=None,
        )
        comparison_rows.append(dict(zip(group_columns, keys, strict=True)) | metrics)
comparison = pd.DataFrame(comparison_rows)
if comparison.empty:
    print("No candidate metrics can be computed yet.")
else:
    display(comparison.sort_values(["position", "target_name", "period", "mae"]))

In [ ]:
if comparison.empty:
    print("The validation-versus-test chart will appear after training.")
else:
    aggregate = comparison.groupby(
        ["candidate_source", "candidate_name", "period"], as_index=False
    ).agg(mean_group_mae=("mae", "mean"))
    chart = aggregate.pivot_table(
        index=["candidate_source", "candidate_name"], columns="period", values="mean_group_mae"
    )
    chart.plot(kind="bar", figsize=(11, 5), title="Candidate comparison by locked period")
    plt.ylabel("Mean of position-target MAE values")
    plt.tight_layout()
    plt.show()

## Read the persisted decision and lineage

The champion record is the decision boundary. The run fingerprints bind that decision to exact features, targets, rules, baseline report, and model contracts. An intact artifact with different lineage is stale, not current.

In [ ]:
if champions.empty:
    print("No champion records exist yet.")
else:
    display(champions)

if runs.empty:
    print("No Phase 4 run lineage exists yet.")
else:
    display(runs)

In [ ]:
if phase4_report:
    report_index = pd.DataFrame(
        [
            {"section": key, "shape_or_value": compact_report_value(value)}
            for key, value in phase4_report.items()
        ]
    )
    display(report_index)
else:
    print("The Phase 4 JSON evaluation report is not present yet.")

## Exercise

Choose one position and target after training. Verify that every required baseline, Ridge, and histogram gradient boosting cover the same 2020-2024 keys. Explain the selected champion from validation only, then describe what 2025 adds without changing that choice. Finally, open the linked model card and state one appropriate use, one inappropriate use, and one lineage fingerprint that would invalidate the result if changed.